# CALM-VAD — multi-dataset evaluation (Colab)

Runs the full CALM-VAD pipeline (M1 reliability → M2 evidence fusion → M3
calibration → M4 alarm budget) and the deployable evaluation harness across
several pose-VAD benchmarks, using **pre-extracted skeletons + ground truth**
published by the community — **no video downloads, no GPU pose extraction**.

| dataset | source | format |
|---|---|---|
| ShanghaiTech | STG-NF release | GEPC json |
| HR-ShanghaiTech / HR-Avenue / HR-UBnormal | MoCoDAD release | Morais csv |

Run cells top-to-bottom. Cell 4 loops over whichever datasets you pick.
A GPU is **not** needed; a CPU runtime is fine.

## Cell 1 — install + code

In [ ]:
!pip -q install numpy scipy scikit-learn pyyaml gdown
REPO_URL = 'https://github.com/FaizanAbbas512/Sentrix.git'   # <- your repo (or '' to upload a zip)
import os, sys
if REPO_URL:
    !rm -rf /content/sentrix && git clone --depth 1 $REPO_URL /content/sentrix
else:
    from google.colab import files
    up = files.upload(); z = next(iter(up))
    !rm -rf /content/sentrix && mkdir -p /content/sentrix && unzip -q "$z" -d /content/sentrix
    s = [d for d in os.listdir('/content/sentrix') if os.path.isdir(f'/content/sentrix/{d}')]
    if 'calm' not in s and len(s) == 1: !cp -r /content/sentrix/{s[0]}/* /content/sentrix/
os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
assert os.path.isdir('calm')
!mkdir -p data/pose data/raw results
!python -m calm.selftest | tail -3

## Cell 2 — dataset registry

`gdrive` ids come from the papers' repos (**verify if a download fails**):
* STG-NF data folder: `github.com/orhir/STG-NF` → README → *Data*
* MoCoDAD poses: `github.com/aleflabo/MoCoDAD` → README → *Data Preparation*

Pick which to run in `RUN`. `hr_*` all come from the one MoCoDAD folder, so it
downloads once.

In [ ]:
DATASETS = {
  'shanghaitech': dict(gdrive='1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt', kind='file',
                       arc='data/raw/stgnf.zip', root='data/raw/stgnf', fps=24),
  'hr_shanghaitech': dict(gdrive='1aUDiyi2FCc6nKTNuhMvpGG_zLZzMMc83', kind='folder',
                          root='data/raw/mocodad', fps=24, sub=('HR-ShanghaiTech','HR-STC','ShanghaiTech')),
  'hr_avenue':    dict(gdrive='1aUDiyi2FCc6nKTNuhMvpGG_zLZzMMc83', kind='folder',
                       root='data/raw/mocodad', fps=25, sub=('HR-Avenue','Avenue','HR_Avenue')),
  'hr_ubnormal':  dict(gdrive='1aUDiyi2FCc6nKTNuhMvpGG_zLZzMMc83', kind='folder',
                       root='data/raw/mocodad', fps=30, sub=('HR-UBnormal','UBnormal','HR_UBnormal')),
}

RUN = ['shanghaitech', 'hr_avenue']    # <- edit this list

# custom link? overwrite an entry, e.g.:
# DATASETS['shanghaitech']['gdrive'] = 'PASTE_ID_OR_LINK'

## Cell 3 — helpers (download + extract, layout-agnostic)

In [ ]:
import os, glob, zipfile, tarfile, gdown, shutil

def _extract_all(folder):
    for p in glob.glob(f'{folder}/**/*', recursive=True):
        if zipfile.is_zipfile(p):
            zipfile.ZipFile(p).extractall(os.path.dirname(p))
        elif p.endswith(('.tar.gz','.tgz','.tar')) and tarfile.is_tarfile(p):
            tarfile.open(p).extractall(os.path.dirname(p))

def fetch(spec):
    root = spec['root']
    if os.path.isdir(root) and glob.glob(f'{root}/**/*.npy', recursive=True):
        return root                                   # already downloaded
    os.makedirs(root, exist_ok=True)
    if spec['kind'] == 'file':
        src = spec['gdrive'] if 'http' in spec['gdrive'] else f"https://drive.google.com/uc?id={spec['gdrive']}"
        gdown.download(src, spec['arc'], quiet=False, fuzzy=True)
        a = spec['arc']
        (zipfile.ZipFile(a).extractall(root) if zipfile.is_zipfile(a)
         else tarfile.open(a).extractall(root))
    else:
        src = spec['gdrive'] if 'http' in spec['gdrive'] else f"https://drive.google.com/drive/folders/{spec['gdrive']}"
        gdown.download_folder(src, output=root, quiet=False, use_cookies=False)
    _extract_all(root)
    return root

def locate(root, subnames):
    """find the dataset subtree (has *.npy masks and json/csv poses)."""
    cands = []
    for s in subnames:
        cands += glob.glob(f'{root}/**/{s}', recursive=True)
    cands = [d for d in cands if os.path.isdir(d)] or [root]
    for d in cands:
        if (glob.glob(f'{d}/**/*.npy', recursive=True) and
            (glob.glob(f'{d}/**/*.json', recursive=True) or glob.glob(f'{d}/**/*.csv', recursive=True))):
            return d
    return cands[0]
print('helpers ready')

## Cell 4 — run every dataset in `RUN`

For each: download (once) → auto-detect layout → full harness → saves
`results/calm_report_<name>.{json,txt}` and a reusable `data/pose/<name>.json`.

In [ ]:
import json, os
done = []
for name in RUN:
    spec = DATASETS[name]
    print('\n' + '=' * 60 + '\n  ' + name + '\n' + '=' * 60)
    root = fetch(spec)
    sub  = locate(root, spec.get('sub', (name,)))
    print('  using subtree:', sub)
    rc = os.system(f"python -m calm.harness --auto '{sub}' --tag {name} "
                   f"--fps {spec['fps']} --save-generic data/pose/{name}.json")
    if rc == 0 and os.path.exists(f'results/calm_report_{name}.json'):
        done.append(name)
    else:
        print('  !! failed - check the subtree / GT printed above')
print('\nfinished:', done)

## Cell 5 — combined comparison table (all datasets, CALM-VAD vs baselines)

In [ ]:
import json, glob, os
rows = []
for jp in sorted(glob.glob('results/calm_report_*.json')):
    r = json.load(open(jp)); ds = r['tag']
    cal = r['calibration']
    for s in r['streams']:
        rows.append(dict(dataset=ds, method=s['label'],
                         AUC=round(s['frame_auc'], 3),
                         eventF1=round(s['event']['f1_avg'], 3),
                         **{f"FAPH@{k.split('=')[-1]}": s[k] for k in s if k.startswith('faph@')}))
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    from IPython.display import display; display(df)
except Exception:
    for x in rows: print(x)

print('\nCalibration (CALM-VAD, ECE raw -> cal) and cost per dataset:')
for jp in sorted(glob.glob('results/calm_report_*.json')):
    r = json.load(open(jp)); c = r['calibration']
    print(f"  {r['tag']:<16} ECE {c['ece_raw']:.4f} -> {c['ece_cal']:.4f}   "
          f"decision {r['cost']['decision_layer_ms_per_frame_mean']:.2f} ms/frame")

## Cell 6 — cross-dataset generalisation (fit on A, test on B)

Uses the `data/pose/<name>.json` files saved in Cell 4.

In [ ]:
import json, itertools, os
PAIRS = list(itertools.permutations([n for n in RUN if os.path.exists(f'data/pose/{n}.json')], 2))
for a, b in PAIRS:
    A = json.load(open(f'data/pose/{a}.json')); B = json.load(open(f'data/pose/{b}.json'))
    for x in A['clips']: x['split'] = 'calib'
    for x in B['clips']: x['split'] = 'test'
    out = f'data/pose/{a}__to__{b}.json'
    json.dump({'fps': A['fps'], 'clips': A['clips'] + B['clips']}, open(out, 'w'))
    os.system(f"python -m calm.harness --generic {out} --tag {a}__to__{b}")
    r = json.load(open(f'results/calm_report_{a}__to__{b}.json'))
    s = [x for x in r['streams'] if 'CALM' in x['label']][0]
    print(f'  {a} -> {b}:  eventF1 {s["event"]["f1_avg"]:.3f}  AUC {s["frame_auc"]:.3f}')

## Cell 7 — download all results + reusable pose files

In [ ]:
!zip -qr /content/calm_all.zip results data/pose/*.json
from google.colab import files; files.download('/content/calm_all.zip')

## Cell 8 (appendix) — GPU path: raw videos → poses

Only if you have raw video files + per-clip GT and want to extract poses
yourself (needs `Runtime → T4 GPU`). Otherwise ignore this cell.

In [ ]:
# !pip -q install ultralytics opencv-python-headless
# TAG = 'myset'
# !python -m calm.extract_poses --videos /content/vids --gt /content/gt \
#     --out data/pose/$TAG.json --split test --weights yolo11n-pose.pt --imgsz 640 --device 0
# !python -m calm.harness --generic data/pose/$TAG.json --tag $TAG